# lipidr: R vs Python parity

Side-by-side comparison of `pylipidr` against the Bioconductor [lipidr](https://bioconductor.org/packages/release/bioc/html/lipidr.html) R reference (Mohamed et al., *J. Proteome Res.* 2020).

The full lipidomics pipeline -- `read_skyline` -> `annotate_lipids` -> `summarize_transitions` -> `normalize_pqn` -> `de_analysis` -> `lsea` -- is run on lipidr's own bundled Skyline example dataset in both languages, so the input is identical.

In [ ]:
import json, subprocess, sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
HERE = Path('.').resolve()

In [ ]:
subprocess.check_call([sys.executable, 'benchmark.py', '--runs', '2'])
summary = json.loads((HERE / 'compare_out' / 'summary.json').read_text())
summary

## 1. Timing

`pylipidr` is pure NumPy/pandas + `pygoslin` (lipid-name parsing) + `pylimma` (moderated-t DE). The biggest win over R is skipping Rscript startup and the heavy Bioconductor `SummarizedExperiment` machinery.

In [ ]:
pd.DataFrame({
    'R lipidr total (ms)':  [summary['r_time_ms']],
    'pylipidr total (ms)':  [summary['py_time_ms']],
    'Speedup':              [f"{summary['speedup']:.2f}x"],
}, index=['Skyline example']).T

## 2. Accuracy

* `annotate_lipids` -- lipid-class agreement vs R lipidr.
* `normalize_pqn` / `normalize_istd` -- Pearson r of normalized intensities.
* `de_analysis` -- Pearson r of logFC and p-values (limma moderated-t).
* `lsea` -- Pearson r of enrichment scores and p-values.

In [ ]:
{k: v for k, v in summary.items() if 'pearson' in k or 'agreement' in k}

In [ ]:
work = HERE / 'compare_out'
r_de = pd.read_csv(work / 'R_out' / 'de.tsv', sep='\t').set_index('Molecule')
import pylipidr as lp
EXTDATA = Path('/scratch/users/steorra/env/CMAP/lib/R/library/lipidr/extdata')
exp = lp.read_skyline(str(EXTDATA / 'A1_data.csv'))
exp = lp.add_sample_annotation(exp, str(EXTDATA / 'clin.csv'))
exp = lp.summarize_transitions(exp, method='max')
norm = lp.normalize_pqn(exp, measure='Area')
de = lp.de_analysis(norm, 'HighFat_water - NormalDiet_water', group_col='group')
py_de = de.set_index('Molecule')
cm = r_de.index.intersection(py_de.index)
fig, ax = plt.subplots(1, 2, figsize=(9, 4.2))
ax[0].scatter(r_de.loc[cm, 'logFC'], py_de.loc[cm, 'logFC'], s=14, alpha=0.6)
lim = [r_de.loc[cm, 'logFC'].min(), r_de.loc[cm, 'logFC'].max()]
ax[0].plot(lim, lim, 'k--', lw=0.8)
ax[0].set_xlabel('R lipidr logFC'); ax[0].set_ylabel('pylipidr logFC')
ax[0].set_title('de_analysis logFC')
ax[1].scatter(-np.log10(r_de.loc[cm, 'P.Value']), -np.log10(py_de.loc[cm, 'P.Value']), s=14, alpha=0.6)
ax[1].set_xlabel('R -log10 P'); ax[1].set_ylabel('pylipidr -log10 P')
ax[1].set_title('de_analysis P.Value')
fig.tight_layout()

## 3. LSEA enrichment scores

Lipid Set Enrichment Analysis (preranked GSEA over class / chain-length / unsaturation sets) -- enrichment scores agree closely; small differences arise because R lipidr's `fgsea` uses an adaptive multilevel permutation scheme while `pylipidr` uses a fixed gene-permutation null.

In [ ]:
r_lsea = pd.read_csv(work / 'R_out' / 'lsea.tsv', sep='\t').set_index('set')
enr = lp.lsea(de, rank_by='logFC', nperm=2000, seed=42).set_index('set')
cm = r_lsea.index.intersection(enr.index)
fig, ax = plt.subplots(figsize=(4.6, 4.6))
ax.scatter(r_lsea.loc[cm, 'ES'], enr.loc[cm, 'ES'], s=30, alpha=0.7)
ax.plot([-1, 1], [-1, 1], 'k--', lw=0.8)
ax.set_xlabel('R lipidr ES'); ax.set_ylabel('pylipidr ES')
ax.set_title('LSEA enrichment score')
fig.tight_layout()